# LOTA vs. LOTA-MP — full experiment run (Kaggle / Colab)

Reproduces every number in the report: trains the LOTA baseline (K=1) and the proposed LOTA-MP (K=4) on Tiny-GenImage and evaluates both on all generators.

**Before running (Kaggle):**
1. *Settings → Accelerator → GPU T4 x2* (or P100) and *Internet → On*.
2. *Add Input →* search `tiny-genimage` (by `yangsangtai`) and add it. If it is not attached, the notebook downloads it with `kagglehub`.
3. *Save Version → Save & Run All (Commit)* runs it in the background; all results appear in the **Output** tab.

On Colab: *Runtime → Change runtime type → T4 GPU*, then *Run all*.

In [ ]:
import os, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO = "https://github.com/SundaramPandey2005/LOTA-AI-Image-Forensics.git"
BRANCH = "main"
os.chdir(WORK)
if not os.path.isdir("lota"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "lota"], check=True)
os.chdir("lota")
!git log --oneline -1
!nvidia-smi -L
!pip install -q -r requirements.txt

## 1. Dataset
Prints the per-generator inventory. Images are used as-is (no resizing or re-encoding).

In [ ]:
out = !python scripts/prepare_data.py --download --data-root /kaggle/input/tiny-genimage
print("
".join(out))
DATA_ROOT = [line for line in out if line.startswith("DATA_ROOT=")][-1].split("=", 1)[1]
OUT = f"{WORK}/outputs"

## 2. Unit tests and a two-minute smoke test of every phase

In [ ]:
!python -m pytest -q tests
!python scripts/run_experiments.py --data-root "$DATA_ROOT" --out "$WORK/outputs_quick" --quick

## 3. Full experiment matrix
Resumable: re-running the cell skips finished jobs. Phases run in priority order (core comparison over 3 seeds → latency → second training generator → robustness → ablations); `--time-budget-hours` stops new jobs before the session limit.

In [ ]:
!python scripts/run_experiments.py --data-root "$DATA_ROOT" --out "$OUT" --time-budget-hours 9

## 4. Result tables

In [ ]:
!python scripts/summarize_results.py --out "$OUT"

## 5. Package the results
`lota_results.zip` holds logs, configs, metrics and per-patch logits (no weights). The checkpoints stay in `outputs/runs/<run>/best.pth`.

In [ ]:
import glob, zipfile
with zipfile.ZipFile(f"{WORK}/lota_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(path) and not path.endswith(".pth"):
            archive.write(path, os.path.relpath(path, WORK))
print("results:", f"{WORK}/lota_results.zip", f"{os.path.getsize(f'{WORK}/lota_results.zip') / 1e6:.1f} MB")
for path in sorted(glob.glob(f"{OUT}/runs/*/best.pth")):
    print(f"checkpoint: {path}  {os.path.getsize(path) / 1e6:.0f} MB")